<a href="https://colab.research.google.com/github/mdsalim0786/AI-ML-Project/blob/main/AI_Customer_Support_Chatbot_Full_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Customer Support Chatbot — Google Colab

Run each code cell from top to bottom. This beginner-to-intermediate project uses Python, Pandas, Scikit-learn, TF-IDF, Logistic Regression, simple sentiment detection and Gradio.

**Demo only:** it does not connect to real orders, payments, refunds or customer accounts. Never enter real passwords, OTPs or payment details.

## 1. Install libraries

In [1]:
!pip -q install -U scikit-learn gradio pandas

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.


## 2. Import libraries

In [2]:
import re
import pandas as pd
import gradio as gr
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


## 3. Create labelled customer-query training data

In [3]:
training_data = {
"greeting": ["hello","hi","hey","good morning","good evening","hello support","I need help","can you help me","is anyone there","good afternoon"],
"order_status": ["where is my order","track my order","order status please","when will my parcel arrive","check my delivery","has my order shipped","tracking is not updating","locate my shipment","where is my package","I have not received my order"],
"delivery_delay": ["my order is delayed","delivery is late","parcel has not arrived on time","shipment is taking too long","expected delivery date has passed","package is stuck in transit","courier has not delivered","why is my shipment late","my parcel is overdue","delivery status says delayed"],
"cancel_order": ["I want to cancel my order","please cancel my purchase","cancel my recent order","I ordered by mistake","stop the shipment","I do not want this order","how can I cancel","cancel the item","withdraw my order","please help cancel my order"],
"return_product": ["I want to return this product","how do I return an item","arrange a product return","what is your return process","send this item back","can I return my purchase","return policy details","I want to exchange this item","how many days to return","return my product"],
"refund": ["I want a refund","please return my money","when will I get my refund","refund has not arrived","how do I claim a refund","refund my cancelled order","money not credited back","check refund status","I am waiting for my refund","process my refund"],
"damaged_product": ["I received a damaged product","item arrived broken","my product is defective","package was damaged","product is not working","I received the wrong item","there is a crack in the product","item was missing","poor product quality","my parcel is damaged"],
"payment_issue": ["my payment failed","money deducted but order not placed","I was charged twice","payment is pending","transaction failed but amount deducted","payment problem","card payment did not work","UPI payment unsuccessful","paid but no order confirmation","amount deducted from my bank"],
"product_information": ["tell me about this product","what are the product features","is this item available","what is the product price","give me product details","available sizes and colours","what services do you offer","explain product specifications","is this in stock","tell me more about the product"],
"account_help": ["I cannot log in","I forgot my password","help reset my password","my account is locked","cannot access my account","update my email address","change my phone number","login is not working","unable to sign in","update my profile"],
"warranty": ["does this product have warranty","how can I claim warranty","my product is under warranty","warranty period details","where can I get product service","repair under warranty","how long is the warranty","is replacement covered","warranty claim process","product warranty information"],
"contact_support": ["how can I contact customer care","give me support email","I want to speak to a human agent","connect me with customer service","reach your support team","customer care number","can support call me","talk to a representative","I need a human","contact customer support"],
"thanks": ["thank you","thanks a lot","that was helpful","thank you for support","great thanks","I appreciate your help","many thanks","very helpful"],
"goodbye": ["bye","goodbye","see you later","that is all","have a nice day","talk to you later","I am done","bye for now"]
}
rows = [{"text": s, "intent": label} for label, examples in training_data.items() for s in examples]
df = pd.DataFrame(rows)
print("Examples:", len(df), "| Intents:", df.intent.nunique())
display(df.sample(10, random_state=7))

Examples: 136 | Intents: 14


,text,intent
135,bye for now,goodbye
108,warranty claim process,warranty
99,update my profile,account_help
65,I received the wrong item,damaged_product
40,I want to return this product,return_product
122,that was helpful,thanks
49,return my product,return_product
74,transaction failed but amount deducted,payment_issue
91,I forgot my password,account_help
132,have a nice day,goodbye


## 4. Explore the dataset

In [4]:
display(df["intent"].value_counts().to_frame("example_count"))

,example_count
intent,
greeting,10
order_status,10
delivery_delay,10
cancel_order,10
return_product,10
refund,10
damaged_product,10
payment_issue,10
product_information,10


## 5. Split data and train the ML model
TF-IDF converts words into numeric features. Logistic Regression learns to classify those features into intents.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    df.text, df.intent, test_size=0.25, random_state=42, stratify=df.intent
)
model = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, strip_accents="unicode",
                              ngram_range=(1, 2), sublinear_tf=True)),
    ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced"))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Holdout accuracy:", round(accuracy_score(y_test, pred), 3))
print(classification_report(y_test, pred, zero_division=0))

Holdout accuracy: 0.706
                     precision    recall  f1-score   support

       account_help       1.00      0.50      0.67         2
       cancel_order       1.00      0.67      0.80         3
    contact_support       1.00      0.50      0.67         2
    damaged_product       0.67      0.67      0.67         3
     delivery_delay       0.50      0.33      0.40         3
            goodbye       0.00      0.00      0.00         2
           greeting       1.00      0.50      0.67         2
       order_status       0.60      1.00      0.75         3
      payment_issue       1.00      0.67      0.80         3
product_information       0.67      1.00      0.80         2
             refund       0.50      1.00      0.67         2
     return_product       0.67      1.00      0.80         2
             thanks       0.50      1.00      0.67         2
           warranty       1.00      1.00      1.00         3

           accuracy                           0.71        3

## 6. Train final model on all labelled examples
The score above is only a small-demo estimate. Use a much larger unseen test set before claiming real-world accuracy.

In [6]:
model.fit(df.text, df.intent)
print("Final model trained.")

Final model trained.


## 7. Create longer support replies
These are prepared response templates selected by the ML prediction. The bot does not know real order or payment status.

In [7]:
responses = {
"greeting": """Hello! Welcome to our Customer Support Centre.

I am your virtual support assistant. I can guide you with order tracking, delayed deliveries, cancellations, returns, refunds, damaged items, payment issues, product information, account access and warranty questions.

Please describe your issue in a sentence. If it concerns an order, you may use a demo reference such as ORD12345. Never share passwords, OTPs, card numbers, CVV or bank details.""",
"order_status": """I can guide you through checking your order status.

1. Sign in to your shopping account.
2. Open My Orders and select the relevant order.
3. Choose Track Order to view the latest shipment update and estimated delivery date.
4. If tracking has not changed for more than 48 hours, contact the seller or courier support with your order reference.

This demo is not connected to a live order database and cannot verify a real shipment. Do you have a demo order reference, such as ORD12345?""",
"delivery_delay": """I am sorry your delivery is taking longer than expected. I understand that waiting for a parcel can be frustrating.

1. Check the latest tracking update in My Orders.
2. Confirm whether the courier has reported a weather, address or operational delay.
3. Verify that your delivery address and phone number are correct.
4. If the estimated date has passed, contact official support with your order reference.

Please tell me the expected delivery date and your demo order reference. I cannot access live courier information.""",
"cancel_order": """I can explain the usual cancellation process.

1. Open My Orders in your account.
2. Select the order you want to cancel.
3. Check whether Cancel Order is available.
4. Select a reason and submit the request.
5. Save the cancellation confirmation.

If the order has already been dispatched, cancellation may not be possible. Check the seller's return policy instead. Is your order still processing or already shipped? This demo cannot cancel a real order.""",
"return_product": """I can guide you through a typical product return.

1. Open My Orders and select the item.
2. Check the return eligibility and return window.
3. Select Return Item and choose a reason.
4. Upload clear product photographs if requested.
5. Follow the pickup or drop-off instructions.
6. Keep the return acknowledgement until the process is complete.

Keep the invoice, packaging and accessories where possible. Is the product unused, defective, damaged or simply unsuitable? This demo cannot create a real return request.""",
"refund": """I understand you want your money back. Here is the usual refund follow-up process.

1. Check whether your cancellation or return was approved.
2. Open the order details and review the refund status.
3. Check the original payment method and bank statement.
4. If the stated refund period has passed, contact billing support with your order and transaction references.
5. Keep the confirmation for follow-up.

Processing time depends on the seller, payment method and bank. This demo cannot initiate or verify a refund. Was the order cancelled, returned, or was money deducted without an order confirmation?""",
"damaged_product": """I am sorry the product arrived damaged, defective or incorrect. That is disappointing.

1. Take clear photographs of the product and outer packaging.
2. Keep the invoice, order confirmation and accessories.
3. Avoid attempting a repair that could affect return or warranty eligibility.
4. Open a return or replacement request through the official order page.
5. Describe the damage and attach photographs if requested.

Was the item physically damaged, defective, or different from what you ordered? This demo cannot approve a replacement or refund.""",
"payment_issue": """I am sorry about the payment issue. Let's identify what happened.

Common situations include:
1. Payment failed and no money was deducted.
2. Money was deducted but no order was created.
3. Payment is still pending.
4. The amount appears to have been charged more than once.

Check your order history and payment-provider statement. If money was deducted without an order confirmation, keep the transaction reference and contact official billing support.

Never share your OTP, UPI PIN, CVV, card PIN or password. This demo cannot view bank transactions or process payments. Which of these situations happened?""",
"product_information": """I can help you understand what to check before purchasing.

Review the product specifications, features, available sizes and colours, price, delivery charges, seller information, stock availability, warranty and return conditions. Customer reviews and product documentation may also help you compare options.

Please share the product name or the feature you want explained. This demo does not have a live product catalogue or current stock information.""",
"account_help": """I can guide you with common account-access problems.

1. Check that your registered email or phone number is correct.
2. Use the official Forgot Password option if needed.
3. Follow the password-reset instructions sent by the platform.
4. Check your internet connection and try again.
5. If your account is locked, contact official support for verification.

Never share your password, OTP or recovery code in this chat. Are you facing a forgotten-password issue, account lock or login error?""",
"warranty": """I can explain the usual warranty claim process.

1. Check the warranty duration and coverage in the product documentation.
2. Keep your invoice and purchase date available.
3. Note the model and serial number if applicable.
4. Contact the authorised service centre or seller.
5. Explain the issue and follow the inspection or repair instructions.

Coverage may exclude accidental damage, misuse or normal wear. Check the manufacturer's actual terms. What product and issue are you asking about? This demo cannot register a real warranty claim.""",
"contact_support": """If you want to speak with a human representative, use the official website or application.

1. Open the Contact Us or Help Centre section.
2. Use only the verified phone number, email or chat channel published there.
3. Explain the issue and provide your support reference if you have one.
4. Ask for a ticket number so you can follow up.

I do not have a verified company phone number or email configured in this demo. Please use the official contact details of the company you purchased from.""",
"thanks": """You are very welcome! I am glad the information helped.

You can ask me about order tracking, delivery delays, returns, cancellations, refunds, payments, damaged products, account access or warranty guidance.

Thank you for contacting our Customer Support Centre.""",
"goodbye": """Thank you for contacting our Customer Support Centre.

I hope the information was useful. Keep your order or support reference handy if you need to follow up with the official support team.

Have a wonderful day!"""
}
print("Prepared replies for", len(responses), "intents.")

Prepared replies for 14 intents.


## 8. Add simple sentiment detection
This is a basic word-list demonstration, not a separately trained sentiment ML model.

In [8]:
positive_words = {"good","great","happy","excellent","thanks","thank","helpful","love","appreciate","wonderful"}
negative_words = {"bad","angry","upset","terrible","poor","late","delayed","broken","damaged","failed","problem","issue","frustrated","disappointed"}

def detect_sentiment(message):
    words = set(re.findall(r"[a-z]+", message.lower()))
    pos, neg = len(words & positive_words), len(words & negative_words)
    if neg > pos: return "negative"
    if pos > neg: return "positive"
    return "neutral"

print(detect_sentiment("I am disappointed because my parcel is late"))
print(detect_sentiment("Thank you, that was helpful"))

negative
positive


## 9. Combine ML prediction, confidence, sentiment and chat history

In [9]:
CONFIDENCE_THRESHOLD = 0.22

def get_reply(message, history=None):
    message = message.strip()
    if not message:
        return "Please type a message so I can help."

    probabilities = model.predict_proba([message])[0]
    index = probabilities.argmax()
    intent = model.classes_[index]
    confidence = float(probabilities[index])
    sentiment = detect_sentiment(message)

    if confidence < CONFIDENCE_THRESHOLD:
        return (
            "I want to make sure I understand correctly. Could you rephrase your question?\n\n"
            "I can help with orders, delivery, cancellation, returns, refunds, damaged items, "
            "payments, product information, account access and warranty questions.\n\n"
            f"Model confidence: {confidence:.1%}"
        )

    reply = responses[intent]
    if sentiment == "negative" and intent not in ["goodbye", "thanks"]:
        reply = "I am sorry you are facing this situation. I understand it can be frustrating. I will guide you through the next steps.\n\n" + reply
    elif sentiment == "positive" and intent not in ["greeting", "thanks", "goodbye"]:
        reply = "Thank you for your message. I am happy to help.\n\n" + reply

    turns = len(history) if history else 0
    reply += (
        f"\n\n--- Chat information ---\n"
        f"Detected category: {intent.replace('_', ' ').title()}\n"
        f"Detected sentiment: {sentiment.title()}\n"
        f"Model confidence: {confidence:.1%}\n"
        f"Previous chat messages: {turns}"
    )
    return reply

print(get_reply("My order is delayed and I am frustrated", []))

I want to make sure I understand correctly. Could you rephrase your question?

I can help with orders, delivery, cancellation, returns, refunds, damaged items, payments, product information, account access and warranty questions.

Model confidence: 14.3%


## 10. Test several messages

In [10]:
test_messages = [
    "Hello, I need help",
    "Can you tell me where my parcel is?",
    "My delivery is late",
    "I want to return a damaged item",
    "Money was deducted but no order was created",
    "How do I reset my password?",
    "Does this product have a warranty?",
    "Thank you for your help"
]
for msg in test_messages:
    print("Customer:", msg)
    print("Predicted intent:", model.predict([msg])[0])
    print("-" * 60)

Customer: Hello, I need help
Predicted intent: greeting
------------------------------------------------------------
Customer: Can you tell me where my parcel is?
Predicted intent: product_information
------------------------------------------------------------
Customer: My delivery is late
Predicted intent: delivery_delay
------------------------------------------------------------
Customer: I want to return a damaged item
Predicted intent: return_product
------------------------------------------------------------
Customer: Money was deducted but no order was created
Predicted intent: payment_issue
------------------------------------------------------------
Customer: How do I reset my password?
Predicted intent: account_help
------------------------------------------------------------
Customer: Does this product have a warranty?
Predicted intent: warranty
------------------------------------------------------------
Customer: Thank you for your help
Predicted intent: thanks
---------

## 11. Launch the web chatbot in Colab using Gradio
Run this cell. Wait for the `gradio.live` link, then open it. Keep the Colab runtime active while using the chatbot.

In [13]:
def chat_function(message, history):
    return get_reply(message, history)

app = gr.ChatInterface(
    fn=chat_function,
    title="AI Customer Support Assistant",
    description="Ask about orders, delivery, returns, refunds, payments, damaged products, accounts or warranty. Learning demo only; no live company systems are connected.",
    examples=[
        "Hello, I need some help",
        "Where is my order?",
        "My delivery is delayed and I am upset",
        "I want to return a damaged product",
        "My payment was deducted but no order was placed",
        "How can I claim a warranty?"
    ]
)
app.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://973e6a18411ab250a8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 12. Optional — Download training data as CSV

In [14]:
df.to_csv("customer_support_training_data.csv", index=False)
from google.colab import files
files.download("customer_support_training_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## How to explain this project in an interview

"I developed an AI-based customer support chatbot using Python, Scikit-learn and Gradio. I prepared a labelled customer-query dataset with multiple support intents, applied TF-IDF vectorization and trained a Logistic Regression classifier to identify customer intent. I added detailed intent-based response templates, basic sentiment detection, confidence-based clarification and conversational history. The chatbot provides an interactive web interface for testing customer queries."

## Limitations and next improvements
- The training dataset is small and illustrative. Add diverse real-world examples and test on unseen data before making performance claims.
- Sentiment detection is rule-based, not a trained sentiment model.
- Replies are prepared templates, not generative AI answers.
- No real order, payment, refund, courier, CRM or customer-account connection exists.
- Gradio's share link and Colab runtime are temporary. Do not enter confidential or personal information.
- Next: connect MySQL for chat/ticket storage, then Power BI for support KPIs; later add an LLM/RAG system for grounded answers from company FAQs.